# 13 — LLMs abertas — zero-shot vs few-shot, binario + multiclasse

Avalia 2 LLMs (Qwen2.5-7B + Llama-3.1-8B) em 2 tarefas (binario + multiclasse top-7+other) com 2 estrategias (zero-shot vs few-shot) = **8 result_cards** no test set.

**Mudanca de prompt**: a partir desta versao os modelos respondem **apenas o label puro** (uma palavra), sem JSON e sem justificativa. Reduz drasticamente o numero de tokens gerados e simplifica o parsing. Parser aceita label puro, JSON-wrapped, e label embutido em frase.

**Few-shot**: 4 exemplos para binario (2 mercado + 2 outros), 8 exemplos para multiclasse (1 por classe). Sampleados do **train set** (nunca do test) com seed fixa para reprodutibilidade. Exemplos persistidos em `runs/llm_few_shot_examples/`.

**Setup necessario**:
1. Aceite a licenca em https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
2. Adicione `HF_TOKEN` nos Colab Secrets (icone de chave > Notebook access ON)
3. L4 24 GB ou A100 (FP16). Em T4 troque LOAD_DTYPE='int8'

**Tempos esperados em L4** (FP16, batch_size=8):
- `MAX_SAMPLES=200` (smoke test): ~30s por combinacao = ~4 min total
- `MAX_SAMPLES=None` (16k): few-shot e ~2x mais lento que zero-shot (prompt maior); estimar ~1h por combinacao = ~8h total

## 0. Verificacao de GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU nao detectada. Va em Runtime > Change runtime type > GPU (L4 ou A100).",
    )

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
print(f"CUDA: {torch.version.cuda}")

HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"

if VRAM_GB < 20:
    print(f"\nAVISO: {GPU_NAME} tem pouca VRAM ({VRAM_GB} GB). "
          f"Considere LOAD_DTYPE='int8'.")


## 1. Bootstrap (Colab + Local)

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    _run(
        [sys.executable, "-m", "pip", "install",
         "transformers>=4.45", "accelerate>=0.34", "-q"],
        "pip install transformers/accelerate",
    )

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "test.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), f"Falta colab_splits.zip em {DRIVE_BASE}."
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")

    RUNS_BASE = DRIVE_BASE / "runs"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"

RUNS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR:", SPLITS_DIR)
print("RUNS_BASE :", RUNS_BASE)
print("HARDWARE  :", HARDWARE)


def _sanity_check_imports() -> None:
    failures = []
    for mod_name in ("numpy", "scipy", "pandas", "torch", "transformers", "economy_classifier"):
        try:
            __import__(mod_name)
        except Exception as e:  # noqa: BLE001
            failures.append((mod_name, type(e).__name__, str(e)))
    if failures:
        for name, kind, msg in failures:
            print(f"  - {name}: {kind}: {msg[:200]}")
        raise RuntimeError(
            "Modulos quebrados. Solucao: Runtime > Restart runtime, e re-execute."
        )
    import torch, transformers  # noqa: E401
    print(f"\ntorch={torch.__version__}  transformers={transformers.__version__}")
    print("OK: economy_classifier importavel")


_sanity_check_imports()


## 2. Autenticacao HuggingFace

Le `HF_TOKEN` dos Colab Secrets (icone de chave na sidebar).

In [ ]:
# Llama-3.1-8B-Instruct e gated. Token lido do Colab Secrets (chave HF_TOKEN).
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("HF_TOKEN carregado de Colab Secrets.")
    except Exception as e:
        print(f"Nao consegui ler HF_TOKEN de userdata: {e}")
        HF_TOKEN = os.environ.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    print("AVISO: HF_TOKEN nao encontrado. Llama-3.1 (gated) vai falhar com GatedRepoError.")
    print("       Qwen2.5 (Apache 2.0) roda sem token.")
else:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Login HuggingFace OK.")


## 3. Imports e configuracao

In [ ]:
import gc
import json
import time
from functools import partial
from pathlib import Path

import pandas as pd
import torch

from economy_classifier.datasets import MULTICLASS_TOP7, OTHER_LABEL, attach_multiclass_label
from economy_classifier.evaluation import (
    compute_binary_metrics,
    compute_confusion_matrix,
    compute_multiclass_metrics,
    compute_cost_metrics,
    compute_roc_auc,
)
from economy_classifier.llm_review import (
    LLM_REGISTRY,
    VALID_BINARY_LABELS,
    VALID_MULTI_LABELS,
    build_few_shot_examples,
    build_review_prompt,
    build_review_prompt_few_shot,
    build_review_prompt_multiclass,
    classify_batch_hf,
    hf_results_to_multiclass_predictions,
    hf_results_to_predictions,
    load_hf_model,
    parse_llm_response,
    parse_llm_response_multiclass,
)
from economy_classifier.project import (
    build_result_card,
    persist_result_card,
)

SEED = 42
MULTI_LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]

# ---- Quais (modelo x tarefa x estrategia) rodar ----
LLMS = [
    ("qwen2.5-7b-instruct",   LLM_REGISTRY["qwen2.5-7b-instruct"]),
    ("llama-3.1-8b-instruct", LLM_REGISTRY["llama-3.1-8b-instruct"]),
]
TASKS = ["binary", "multiclass"]
STRATEGIES = ["zero_shot", "few_shot"]

# ---- Few-shot config ----
N_PER_CLASS_BINARY = 2   # 2 mercado + 2 outros = 4 exemplos no prompt
N_PER_CLASS_MULTI  = 1   # 1 por classe x 8 classes = 8 exemplos
EXAMPLE_MAX_CHARS  = 500  # truncamento dos textos de demonstracao

# ---- Inferencia ----
LOAD_DTYPE       = "float16"
BATCH_SIZE       = 8
MAX_NEW_TOKENS   = 16          # resposta = 1 palavra (label puro), 16 tokens com folga
MAX_INPUT_LENGTH = 3072        # few-shot multiclasse pode chegar a ~2500 tokens
TEMPERATURE      = 0.0

# ---- Sampling do test set ----
# 200 = smoke test rapido. Para producao, troque por None ou 16629.
MAX_SAMPLES = 200

print(f"Combinacoes: {len(LLMS)} LLMs x {len(TASKS)} tarefas x {len(STRATEGIES)} estrategias "
      f"= {len(LLMS) * len(TASKS) * len(STRATEGIES)} cards")
print(f"BATCH_SIZE  : {BATCH_SIZE}")
print(f"MAX_SAMPLES : {MAX_SAMPLES}")


## 4. Carga dos splits

In [ ]:
train = pd.read_parquet(SPLITS_DIR / "train.parquet")
test = pd.read_parquet(SPLITS_DIR / "test.parquet")
if "label_multi" not in train.columns:
    train = attach_multiclass_label(train)
if "label_multi" not in test.columns:
    test = attach_multiclass_label(test)

print(f"train: {len(train):,}  test: {len(test):,}  (mercado em test={test['label'].mean()*100:.2f}%)")

if MAX_SAMPLES is not None and MAX_SAMPLES < len(test):
    test_eval = (
        test.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(n=int(MAX_SAMPLES * len(g) / len(test)),
                                       random_state=SEED))
            .reset_index(drop=False)
    )
    test_eval = test_eval.set_index("index")
    print(f"Subsample: {len(test_eval):,} registros (mercado={test_eval['label'].mean()*100:.2f}%)")
else:
    test_eval = test.copy()

RECORD_IDS  = test_eval.index.tolist()
TEXTS       = test_eval["text"].fillna("").tolist()
Y_TRUE_BIN  = test_eval["label"].values
Y_TRUE_MULTI = test_eval["label_multi"].values


## 5. Sample dos exemplos few-shot do TRAIN set

Sampleados uma vez (seed fixa) e persistidos para auditoria. Os mesmos exemplos sao reusados em todas as execucoes.

In [ ]:
# Sample exemplos do TRAIN set (nunca do test) para uso few-shot.
# Mesmo seed -> mesmos exemplos toda execucao -> reprodutivel.

EXAMPLES_BINARY = build_few_shot_examples(
    train,
    label_column="label",
    valid_labels=VALID_BINARY_LABELS,
    n_per_class=N_PER_CLASS_BINARY,
    text_max_chars=EXAMPLE_MAX_CHARS,
    seed=SEED,
)
print(f"Exemplos few-shot BINARIO: {len(EXAMPLES_BINARY)} ("
      f"{sum(1 for _, l in EXAMPLES_BINARY if l == 'mercado')} mercado, "
      f"{sum(1 for _, l in EXAMPLES_BINARY if l == 'outros')} outros)")

EXAMPLES_MULTI = build_few_shot_examples(
    train,
    label_column="label_multi",
    valid_labels=VALID_MULTI_LABELS,
    n_per_class=N_PER_CLASS_MULTI,
    text_max_chars=EXAMPLE_MAX_CHARS,
    seed=SEED,
)
labels_multi = [l for _, l in EXAMPLES_MULTI]
print(f"Exemplos few-shot MULTICLASSE: {len(EXAMPLES_MULTI)} "
      f"(classes: {sorted(set(labels_multi))})")

# Persistir os exemplos usados para auditoria
EXAMPLES_DIR = RUNS_BASE / "llm_few_shot_examples"
EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)
(EXAMPLES_DIR / "examples_binary.json").write_text(json.dumps(
    [{"text": t, "label": l} for t, l in EXAMPLES_BINARY], indent=2,
))
(EXAMPLES_DIR / "examples_multiclass.json").write_text(json.dumps(
    [{"text": t, "label": l} for t, l in EXAMPLES_MULTI], indent=2,
))
print(f"Exemplos persistidos em {EXAMPLES_DIR}")


## 6. Funcao de avaliacao por (modelo, tarefa, estrategia)

`evaluate_llm(model_key, model_name, tokenizer, model, task, strategy)`:
- `strategy='zero_shot'`: prompt = system + query
- `strategy='few_shot'`: prompt = system + N exemplos (user, assistant) + query

Modelo e tokenizer carregados uma vez por LLM e reusados nas 4 combinacoes (task x strategy) para evitar 4 cargas redundantes.

In [ ]:
def evaluate_llm(
    model_key: str, model_name: str,
    tokenizer, model,
    task: str, strategy: str,
) -> dict:
    """Classifica test_eval para uma combinacao (task, strategy).

    O modelo e tokenizer ja vem carregados (compartilhados entre as 4 combinacoes
    do mesmo LLM, evitando carregar 4x).
    """
    if task not in {"binary", "multiclass"}:
        raise ValueError(f"task invalida: {task}")
    if strategy not in {"zero_shot", "few_shot"}:
        raise ValueError(f"strategy invalida: {strategy}")

    model_id = f"llm_{model_key.replace('.', '_').replace('-', '_')}"
    run_dir_name = f"{model_id}_{task}_{strategy}_test_set"
    run_dir = RUNS_BASE / run_dir_name
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / "predictions_checkpoint.csv"

    print(f"\n--- {model_id}  task={task}  strategy={strategy} ---")

    # ---- Selecao de prompt builder + parser ----
    multiclass = (task == "multiclass")
    if strategy == "zero_shot":
        prompt_builder = (
            build_review_prompt_multiclass if multiclass else build_review_prompt
        )
    else:  # few_shot
        examples = EXAMPLES_MULTI if multiclass else EXAMPLES_BINARY
        prompt_builder = partial(
            build_review_prompt_few_shot, examples=examples, multiclass=multiclass,
        )

    parser = parse_llm_response_multiclass if multiclass else parse_llm_response
    Y_TRUE = Y_TRUE_MULTI if multiclass else Y_TRUE_BIN

    # ---- Inferencia ----
    t0 = time.perf_counter()
    raw_results = classify_batch_hf(
        tokenizer, model,
        texts=TEXTS, record_ids=RECORD_IDS,
        method=f"{model_key}_{strategy}",
        max_new_tokens=MAX_NEW_TOKENS,
        max_input_length=MAX_INPUT_LENGTH,
        temperature=TEMPERATURE,
        batch_size=BATCH_SIZE,
        checkpoint_path=checkpoint,
        checkpoint_every=200,
        prompt_builder=prompt_builder,
        parser=parser,
    )
    inference_time = time.perf_counter() - t0
    n_errors = sum(1 for r in raw_results if r.get("label") == "erro")
    print(f"  -> {len(raw_results)} predicoes em {inference_time:.0f}s "
          f"({n_errors} erros de parse)")

    # ---- Predictions DataFrame + metricas ----
    if multiclass:
        preds = hf_results_to_multiclass_predictions(raw_results)
        truth = pd.DataFrame({"index": RECORD_IDS, "y_true": Y_TRUE})
        preds = preds.merge(truth, on="index", how="left")
        preds = preds[["index", "y_true", "y_pred", "method", "label"]]
        valid = preds.dropna(subset=["y_true"])
        metrics = compute_multiclass_metrics(
            valid["y_true"], valid["y_pred"], labels=MULTI_LABELS,
        )
        metrics["coverage"] = round(len(valid) / len(test_eval), 4)
        cm = compute_confusion_matrix(valid["y_true"], valid["y_pred"], labels=MULTI_LABELS)
        cm.to_csv(run_dir / "confusion_matrix.csv")
        primary = metrics["macro_f1"]
        print(f"  macro_f1={primary:.4f}  acc={metrics['accuracy']:.4f}  cov={metrics['coverage']:.4f}")
    else:
        preds = hf_results_to_predictions(raw_results)
        truth = pd.DataFrame({"index": RECORD_IDS, "y_true": Y_TRUE})
        preds = preds.merge(truth, on="index", how="left")
        preds = preds[["index", "y_true", "y_pred", "y_score", "method", "label"]]
        valid = preds.dropna(subset=["y_true"])
        metrics = compute_binary_metrics(valid["y_true"].values, valid["y_pred"].values)
        metrics["auc_roc"] = round(
            compute_roc_auc(valid["y_true"].values, valid["y_score"].values), 4,
        )
        metrics["coverage"] = round(len(valid) / len(test_eval), 4)
        primary = metrics["f1"]
        print(f"  f1={primary:.4f}  prec={metrics['precision']:.4f}  "
              f"rec={metrics['recall']:.4f}  cov={metrics['coverage']:.4f}")

    preds.to_csv(run_dir / "predictions.csv", index=False)

    n_params = sum(p.numel() for p in model.parameters())
    model_size_mb = round(n_params * (2 if LOAD_DTYPE == "float16" else 1) / 1e6, 1)

    cost = compute_cost_metrics(
        train_seconds=0.0,
        inference_seconds=inference_time,
        n_inference_samples=len(test_eval),
        model_size_mb=model_size_mb,
        n_parameters=int(n_params),
        hardware=HARDWARE,
    )
    cost["batch_size"] = BATCH_SIZE
    cost["max_new_tokens"] = MAX_NEW_TOKENS
    cost["strategy"] = strategy
    if strategy == "few_shot":
        cost["n_examples"] = len(EXAMPLES_MULTI if multiclass else EXAMPLES_BINARY)

    config = {
        "model_name": model_name,
        "dtype": LOAD_DTYPE,
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_input_length": MAX_INPUT_LENGTH,
        "temperature": TEMPERATURE,
        "max_samples": MAX_SAMPLES,
        "task": task,
        "strategy": strategy,
    }

    persist_result_card(build_result_card(
        model_id=model_id, task=task, regime="test_set",
        metrics=metrics, cost=cost, config=config,
        n_train_samples=0, n_eval_samples=len(test_eval),
        predictions_path=str(run_dir / "predictions.csv"),
        notes=(f"Zero/few-shot LLM no test set. strategy={strategy}. "
               "Resposta = label puro (sem JSON, sem justificativa)."),
    ), run_dir)

    return {
        "model_id": model_id, "task": task, "strategy": strategy,
        "metrics": metrics, "cost": cost, "n_errors": n_errors, "primary": primary,
    }


## 7. Loop sobre todas as combinacoes

Estrutura: para cada LLM, carrega modelo uma vez, depois itera sobre as 4 (task, strategy). Falhas isoladas nao matam o loop.

In [ ]:
all_summaries = []

for model_key, model_name in LLMS:
    print(f"\n{'='*72}\nLLM: {model_key}  ({model_name})\n{'='*72}")
    try:
        t0 = time.perf_counter()
        tokenizer, model = load_hf_model(model_name, dtype=LOAD_DTYPE)
        load_time = time.perf_counter() - t0
        print(f"Modelo carregado em {load_time:.1f}s.")
    except Exception as exc:
        print(f"FALHA AO CARREGAR {model_key}: {type(exc).__name__}: {exc}")
        print("Pulando todas as combinacoes deste modelo.")
        continue

    for task in TASKS:
        for strategy in STRATEGIES:
            try:
                summary = evaluate_llm(
                    model_key, model_name,
                    tokenizer, model,
                    task=task, strategy=strategy,
                )
                all_summaries.append(summary)
            except Exception as exc:
                print(f"FALHA em {model_key}/{task}/{strategy}: {type(exc).__name__}: {exc}")
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    # Liberar VRAM antes do proximo modelo
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n\n=== TODAS AS COMBINACOES AVALIADAS ===")
for s in all_summaries:
    print(f"{s['model_id']:35s} task={s['task']:10s} strategy={s['strategy']:10s} "
          f"primary={s['primary']:.4f}  errors={s['n_errors']}  cov={s['metrics'].get('coverage'):.4f}")


## 8. Sumario comparativo (8 cards)

In [ ]:
rows = []
for sub in sorted(RUNS_BASE.glob("llm_*_test_set")):
    card_path = sub / "result_card.json"
    if not card_path.exists():
        continue
    c = json.loads(card_path.read_text())
    if c.get("task") not in ("binary", "multiclass"):
        continue
    m = c["metrics"]
    cfg = c.get("config", {})
    primary = m.get("f1") or m.get("macro_f1")
    rows.append({
        "model_id": c["model_id"],
        "task": c["task"],
        "strategy": cfg.get("strategy", "zero_shot"),
        "primary": primary,
        "coverage": m.get("coverage"),
        "inf_s": c["cost"].get("inference_seconds_total") or c["cost"].get("inference_seconds_mean"),
        "size_mb": c["cost"].get("model_size_mb"),
        "params_b": round((c["cost"].get("n_parameters") or 0) / 1e9, 2),
    })
summary_df = pd.DataFrame(rows).sort_values(
    ["task", "strategy", "primary"], ascending=[True, True, False],
).reset_index(drop=True)
summary_df
